In [16]:
import pandas as pd

In [17]:
df = pd.read_csv("../data/cleaned_results_2000.csv")

In [18]:
df.shape

(25220, 11)

In [19]:
def get_previous_matches(team, match_date, n=5):

    matches = team_matches[team]

    previous = matches[
        matches["date"] < match_date
    ]

    return previous.tail(n)

In [20]:
def calculate_form(team, match_date):

    previous = get_previous_matches(team, match_date)

    if len(previous) < 5:
        return None

    points = 0

    for _, row in previous.iterrows():

        if row["home_team"] == team:

            if row["home_score"] > row["away_score"]:
                points += 3

            elif row["home_score"] == row["away_score"]:
                points += 1

        else:

            if row["away_score"] > row["home_score"]:
                points += 3

            elif row["away_score"] == row["home_score"]:
                points += 1

    return points / 15

In [21]:
def avg_goal_difference(team, match_date):

    previous = get_previous_matches(team, match_date)

    if len(previous) < 5:
        return None

    diffs = []

    for _, row in previous.iterrows():

        if row["home_team"] == team:
            diff = row["home_score"] - row["away_score"]
        else:
            diff = row["away_score"] - row["home_score"]

        diffs.append(diff)

    return sum(diffs) / len(diffs)

In [22]:
def avg_goals_scored(team, match_date):

    previous = get_previous_matches(team, match_date)

    if len(previous) < 5:
        return None

    goals = []

    for _, row in previous.iterrows():

        if row["home_team"] == team:
            goals.append(row["home_score"])
        else:
            goals.append(row["away_score"])

    return sum(goals) / len(goals)

In [23]:
def avg_goals_conceded(team, match_date):

    previous = get_previous_matches(team, match_date)

    if len(previous) < 5:
        return None

    goals = []

    for _, row in previous.iterrows():

        if row["home_team"] == team:
            goals.append(row["away_score"])
        else:
            goals.append(row["home_score"])

    return sum(goals) / len(goals)

In [24]:
def expected_score(rating_a, rating_b):

    return 1 / (
        1 + 10 ** ((rating_b - rating_a) / 400)
    )

In [25]:
def update_elo(
    rating_a,
    rating_b,
    score_a,
    k=20
):

    expected_a = expected_score(
        rating_a,
        rating_b
    )

    return rating_a + k * (
        score_a - expected_a
    )

def get_tournament_category(tournament):

    t = tournament.lower()

    # ------------------------
    # World Cup
    # ------------------------

    if t == "fifa world cup":
        return "world_cup"

    elif "world cup qualification" in t:
        return "world_cup_qualifier"

    # ------------------------
    # Nations League
    # ------------------------

    elif "nations league" in t:
        return "nations_league"

    # ------------------------
    # Continental Championships
    # ------------------------

    elif t in [

        "uefa euro",
        "copa américa",
        "african cup of nations",
        "afc asian cup",
        "gold cup",

        "confederations cup"

    ]:
        return "continental"

    # ------------------------
    # Continental Qualifiers
    # ------------------------

    elif any(x in t for x in [

        "euro qualification",

        "african cup of nations qualification",

        "asian cup qualification",

        "gold cup qualification",

        "copa américa qualification"

    ]):
        return "continental_qualifier"

    # ------------------------
    # Friendly
    # ------------------------

    elif t == "friendly":
        return "friendly"

    # ------------------------
    # Regional Competitions
    # ------------------------

    elif any(x in t for x in [

        "cecafa",
        "cosafa",
        "cfu",
        "gulf",
        "saff",
        "eaff",
        "waff",
        "uncaf",
        "conifa",
        "island games",
        "arab cup",
        "challenge cup"

    ]):
        return "regional"

    # ------------------------
    # Everything Else
    # ------------------------

    else:
        return "other"

In [26]:
elo_ratings = {}

all_teams = set(df["home_team"]).union(
    set(df["away_team"])
)

for team in all_teams:
    elo_ratings[team] = 1500

home_elos = []
away_elos = []
elo_diffs = []

for _, match in df.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]

    home_rating = elo_ratings[home_team]
    away_rating = elo_ratings[away_team]

    home_elos.append(home_rating)
    away_elos.append(away_rating)

    elo_diffs.append(
        home_rating - away_rating
    )

    if match["home_score"] > match["away_score"]:
        home_score = 1
        away_score = 0

    elif match["home_score"] < match["away_score"]:
        home_score = 0
        away_score = 1

    else:
        home_score = 0.5
        away_score = 0.5

    elo_ratings[home_team] = update_elo(
        home_rating,
        away_rating,
        home_score
    )

    elo_ratings[away_team] = update_elo(
        away_rating,
        home_rating,
        away_score
    )

df["home_elo"] = home_elos
df["away_elo"] = away_elos
df["elo_diff"] = elo_diffs

In [27]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,result,home_elo,away_elo,elo_diff
0,2000-01-04,Egypt,Togo,2,1,Friendly,Aswan,Egypt,False,2000,home_win,1500.0,1500.0,0.0
1,2000-01-07,Tunisia,Togo,7,0,Friendly,Tunis,Tunisia,False,2000,home_win,1500.0,1490.0,10.0
2,2000-01-08,Trinidad and Tobago,Canada,0,0,Friendly,Port of Spain,Trinidad and Tobago,False,2000,draw,1500.0,1500.0,0.0
3,2000-01-09,Burkina Faso,Gabon,1,1,Friendly,Ouagadougou,Burkina Faso,False,2000,draw,1500.0,1500.0,0.0
4,2000-01-09,Guatemala,Armenia,1,1,Friendly,Los Angeles,United States,True,2000,draw,1500.0,1500.0,0.0


In [28]:
team_matches = {}

for team in set(df["home_team"]).union(set(df["away_team"])):

    matches = pd.concat([
        df[df["home_team"] == team],
        df[df["away_team"] == team]
    ])

    team_matches[team] = matches.sort_values("date")

In [41]:
feature_rows = []

for _, match in df.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]

    match_date = match["date"]

    home_form = calculate_form(
        home_team,
        match_date
    )

    away_form = calculate_form(
        away_team,
        match_date
    )

    home_gd = avg_goal_difference(
        home_team,
        match_date
    )

    away_gd = avg_goal_difference(
        away_team,
        match_date
    )

    home_scored = avg_goals_scored(
        home_team,
        match_date
    )

    away_scored = avg_goals_scored(
        away_team,
        match_date
    )

    home_conceded = avg_goals_conceded(
        home_team,
        match_date
    )

    away_conceded = avg_goals_conceded(
        away_team,
        match_date
    )

    if (
        home_form is None or
        away_form is None or
        home_gd is None or
        away_gd is None or
        home_scored is None or
        away_scored is None or
        home_conceded is None or
        away_conceded is None
    ):
        continue

    feature_rows.append({

        "form_diff":
            home_form - away_form,

        "goal_diff_diff":
            home_gd - away_gd,

        "scored_diff":
            home_scored - away_scored,

        "conceded_diff":
            home_conceded - away_conceded,

        "elo_diff":
            match["elo_diff"],
        "neutral": int(match["neutral"]),

        "tournament_type":get_tournament_category(match["tournament"]),

        "year":match["year"],

        "result":
            match["result"]

    })

In [42]:
features_df = pd.DataFrame(feature_rows)

print(features_df.shape)

features_df.head()

(24147, 9)


,form_diff,goal_diff_diff,scored_diff,conceded_diff,elo_diff,neutral,tournament_type,year,result
0,0.200000,0.4,-0.2,-0.6,10.353400,1,continental,2000,home_win
1,0.133333,-0.4,-0.6,-0.2,19.145917,1,continental,2000,away_win
2,0.066667,1.0,0.4,-0.6,-1.107338,1,continental,2000,home_win
3,0.200000,1.0,0.4,-0.6,18.005513,1,continental,2000,draw
4,0.200000,0.2,0.2,0.0,19.297593,0,continental,2000,draw


In [50]:
features_df.to_csv(
    "../data/features_2000B.csv",
    index=False
)

In [43]:
tournament_counts = (
    df["tournament"]
    .value_counts()
)

print(tournament_counts.head(100))

tournament
Friendly                                8340
FIFA World Cup qualification            5840
UEFA Euro qualification                 1532
African Cup of Nations qualification    1416
UEFA Nations League                      658
                                        ... 
OSN Cup                                    4
Copa América qualification                 4
Navruz Cup                                 4
CONIFA Africa Football Cup                 4
Jordan International Tournament            4
Name: count, Length: 100, dtype: int64


In [33]:
df["tournament"].value_counts().head(30)

tournament
Friendly                                8340
FIFA World Cup qualification            5840
UEFA Euro qualification                 1532
African Cup of Nations qualification    1416
UEFA Nations League                      658
AFC Asian Cup qualification              530
African Cup of Nations                   525
CONCACAF Nations League                  422
FIFA World Cup                           384
Gold Cup                                 359
CECAFA Cup                               341
COSAFA Cup                               326
CFU Caribbean Cup qualification          293
Island Games                             283
UEFA Euro                                277
AFC Asian Cup                            256
AFF Championship                         251
Copa América                             248
Gulf Cup                                 189
SAFF Cup                                 135
EAFF Championship                        130
WAFF Championship                        114

In [44]:
features_df = pd.get_dummies(
    features_df,
    columns=["tournament_type"],
    drop_first=True
)

In [45]:
features_df

,form_diff,goal_diff_diff,scored_diff,conceded_diff,elo_diff,neutral,year,result,tournament_type_continental_qualifier,tournament_type_friendly,tournament_type_nations_league,tournament_type_other,tournament_type_regional,tournament_type_world_cup,tournament_type_world_cup_qualifier
0,0.200000,0.4,-0.2,-0.6,10.353400,1,2000,home_win,False,False,False,False,False,False,False
1,0.133333,-0.4,-0.6,-0.2,19.145917,1,2000,away_win,False,False,False,False,False,False,False
2,0.066667,1.0,0.4,-0.6,-1.107338,1,2000,home_win,False,False,False,False,False,False,False
3,0.200000,1.0,0.4,-0.6,18.005513,1,2000,draw,False,False,False,False,False,False,False
4,0.200000,0.2,0.2,0.0,19.297593,0,2000,draw,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24142,0.066667,1.0,0.8,-0.2,-5.580990,0,2026,away_win,False,True,False,False,False,False,False
24143,0.133333,1.6,0.6,-1.0,169.982580,0,2026,home_win,False,True,False,False,False,False,False
24144,-0.133333,0.2,0.0,-0.2,87.990287,0,2026,home_win,False,True,False,False,False,False,False
24145,-0.200000,-0.4,0.8,1.2,-46.797629,0,2026,home_win,False,True,False,False,False,False,False


In [46]:
features_df.head()

,form_diff,goal_diff_diff,scored_diff,conceded_diff,elo_diff,neutral,year,result,tournament_type_continental_qualifier,tournament_type_friendly,tournament_type_nations_league,tournament_type_other,tournament_type_regional,tournament_type_world_cup,tournament_type_world_cup_qualifier
0,0.200000,0.4,-0.2,-0.6,10.353400,1,2000,home_win,False,False,False,False,False,False,False
1,0.133333,-0.4,-0.6,-0.2,19.145917,1,2000,away_win,False,False,False,False,False,False,False
2,0.066667,1.0,0.4,-0.6,-1.107338,1,2000,home_win,False,False,False,False,False,False,False
3,0.200000,1.0,0.4,-0.6,18.005513,1,2000,draw,False,False,False,False,False,False,False
4,0.200000,0.2,0.2,0.0,19.297593,0,2000,draw,False,False,False,False,False,False,False


In [47]:
features_df.filter(
    regex="tournament_type"
).sum()

tournament_type_continental_qualifier    3479
tournament_type_friendly                 7947
tournament_type_nations_league           1148
tournament_type_other                    1403
tournament_type_regional                 2396
tournament_type_world_cup                 384
tournament_type_world_cup_qualifier      5684
dtype: int64

In [48]:
dummy_cols = features_df.filter(
    regex="tournament_type"
).columns

features_df[dummy_cols] = (
    features_df[dummy_cols]
    .astype(int)
)

In [49]:
features_df[dummy_cols].head()

,tournament_type_continental_qualifier,tournament_type_friendly,tournament_type_nations_league,tournament_type_other,tournament_type_regional,tournament_type_world_cup,tournament_type_world_cup_qualifier
0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0


In [51]:
features_df.columns.tolist()

['form_diff',
 'goal_diff_diff',
 'scored_diff',
 'conceded_diff',
 'elo_diff',
 'neutral',
 'year',
 'result',
 'tournament_type_continental_qualifier',
 'tournament_type_friendly',
 'tournament_type_nations_league',
 'tournament_type_other',
 'tournament_type_regional',
 'tournament_type_world_cup',
 'tournament_type_world_cup_qualifier']